# Industry Spillover Risk Pipeline

## 1.0 – Global Configuration, Imports & Paths
Set `MARKET` to `'china'`, `'us'`, or iterate both. All modules share this namespace.

Run this in terminal before importing:

```bash
pip install numpy ipykernel
pip install pandas ipykernel
pip install statsmodels ipykernel
pip install arch ipykernel
pip install openpyxl
```

### 1.1 – Import required files

- Imports, warning suppressions, and logging configuration
- Wires up the libraries used across the spillover pipeline:
    - numpy & pandas: data handling
    - arch: GARCH & EGARCH volatility models
    - statsmodels: VAR & ADF analysis (Augmented Dickey Fuller, is a common statistical test used to test whether a given Time series is stationary or not)
- Warnings are silenced globally since arch_model() raises noisy optimizer warnings that are already captured and logged via _log_issue() instead of being surfaced as Python warnings
- NFO-level logging: every pipeline stage logs a timestamped status line to the notebook output, which doubles as a lightweight audit trail of each run.

In [14]:
import os, pickle, logging, traceback, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from arch import arch_model
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
warnings.filterwarnings('ignore')

# NFO-level logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s  %(levelname)-8s  %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S')
log = logging.getLogger(__name__)



### 1.2 – Paths

Resolve the project's directory layout relative to the notebook's location and make sure the `outputs/` and `logs/` directories exist before any later stage attempts to write a file into them.

In [15]:
BASE_DIR    = Path().resolve()            # notebook(base) directory
DATA_DIR    = BASE_DIR / 'data'
OUTPUTS_DIR = BASE_DIR / 'outputs'
LOGS_DIR    = BASE_DIR / 'logs'
ISSUE_LOG   = LOGS_DIR / 'issue_log.txt'
for d in [OUTPUTS_DIR, LOGS_DIR]: d.mkdir(parents=True, exist_ok=True)

### 1.3 – Hyperparameters

Global hyperparameters shared by these stages:
- GARCH/EGARCH fitting
- ADF stationarity testing
- VAR lag selection
- Diebold-Yilmaz GFEVD / rolling-connectedness analysis

Centralising them here means switching MARKET (or any threshold) reruns the whole pipeline consistently.

In [16]:
# ---------- MARKETS TO PROCESS ----------
MARKETS         = ['china', 'us'] # Will process both countries sequentially
# ----------------------------------------
JUMP_THRESHOLD  = 20.0        # flag |R| > 20%
HORIZON         = 10          # GFEVD forecast horizon (DY 2014)
ROLLING_WINDOW  = 200         # rolling TSI window (trading days)
ROLLING_STEP    = 1           # step size
MAX_LAGS        = 10          # AIC evaluation bound
ADF_MAXLAG      = 12
ADF_PVAL_THR    = 0.05
PERSISTENCE_LOW  = 0.95
PERSISTENCE_HIGH = 0.99
INNOVATION_DIST  = 't'        # Student-t innovations
MAX_ITER         = 200
MAX_ITER_RETRY   = 500

# === NEW GRAPH CONNECTION PARAMETER ===
TOP_N_CONNS      = 5          # Extracted top-connected edges per window

GICS_SECTORS = [
    'Energy', 'Materials', 'Industrials',
    'Consumer Discretionary', 'Consumer Staples', 'Health Care',
    'Financials', 'Information Technology', 'Communication Services',
    'Utilities', 'Real Estate',
]

### 1.4 – Market file configs
*** Update paths/columns as necessary

Per-market schema configuration: maps each raw data file's column names and raw sector codes/labels onto the canonical GICS_SECTORS names used by every downstream stage. 

In [17]:
MARKET_CONFIGS = {
    'china': {
        'file': 'China Industry Index.xlsx',
        'date_col': 'Date',
        'price_col': 'Closeprice',
        'multi_sheet': False,
        'benchmark_col': None,
        'sector_map': {
            986: 'Energy',
            987: 'Materials',
            988: 'Industrials',
            989: 'Consumer Discretionary',
            990: 'Consumer Staples',
            991: 'Health Care',
            993: 'Information Technology',
            994: 'Communication Services',
            995: 'Utilities',
            931775: 'Real Estate',
            932075: 'Financials'
        },
        'benchmark_col': 300
    },
    'us': {
        'file': 'US industry index.xlsx',
        'date_col': 'Name',
        'price_col': None,
        'multi_sheet': False,
        'benchmark_col': 'S&P 500 COMPOSITE - PRICE INDEX',
        'sector_map': {
            'S&P500 ES ENERGY - PRICE INDEX':                 'Energy',
            'S&P500 ES MATERIALS - PRICE INDEX':              'Materials',
            'S&P500 ES INDUSTRIALS - PRICE INDEX':            'Industrials',
            'S&P500 ES CONSUMER DISCRETIONARY - PRICE INDEX': 'Consumer Discretionary',
            'S&P500 ES CONSUMER STAPLES - PRICE INDEX':       'Consumer Staples',
            'S&P500 ES HEALTH CARE - PRICE INDEX':            'Health Care',
            'S&P500 ES FINANCIALS - PRICE INDEX':             'Financials',
            'S&P500 ES REAL ESTATE - PRICE INDEX':            'Real Estate',
            'S&P500 ES INFO TECHNOLOGY - PRICE INDEX':        'Information Technology',
            'S&P500 ES COMM. SVS - PRICE INDEX':              'Communication Services',
            'S&P500 ES UTILITIES - PRICE INDEX':              'Utilities',
        },
    },
}

### 1.5 – Load market config

Append a timestamped diagnostic entry to the issue log and emit a warning.

Used throughout the pipeline (data-quality flags, ADF non-stationarity, GARCH/EGARCH convergence problems, rolling-window failures, etc.) to record non-fatal problems for later review without halting execution.

Args:
- market (str): Market identifier (e.g. 'china', 'us'), used to tag the log entry
- context (str): Short label identifying which pipeline stage raised the issue (e.g. 'JUMP', 'ADF', 'LAG_SELECTION', 'ROLLING')
- message (str): Human-readable description of the issue

Returns:
- None: nothing is returned, writes a line to ISSUE_LOG and emits a logger warning as a side effect

In [18]:
def _log_issue(market, context, message):
    ts    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f'[{ts}] [{market.upper()}] [{context}] {message}\n'
    with open(ISSUE_LOG, 'a') as fh: fh.write(entry)
    log.warning(entry.strip())

log.info('Configuration loaded. MARKET=%s', MARKETS)

2026-06-25 14:01:25  INFO      Configuration loaded. MARKET=['china', 'us']


## 2.0 – Data Preprocessing & Return Construction

In [19]:
# --- Section 2.1: Updated Dynamic Data Loader ---
for MARKET in MARKETS:
    cfg = MARKET_CONFIGS[MARKET]
    filepath = DATA_DIR / MARKET / cfg['file']

    # 1. Load data
    if filepath.suffix in ('.xlsx', '.xls'):
        df_raw = pd.read_excel(filepath)
    else:
        df_raw = pd.read_csv(filepath)

    # 2. Identify if the data is Long (China) or Wide (US)
    # We check if the expected 'sector_col' or 'Index' exists. 
    # If not, we assume it is already in wide format.
    if 'Index' in df_raw.columns:
        # --- CHINA FORMAT (LONG) ---
        df_raw[cfg['date_col']] = pd.to_datetime(df_raw[cfg['date_col']], errors='coerce')
        
        # Use the mapping to pivot correctly
        price_panel = df_raw.groupby([cfg['date_col'], 'Index'])[cfg['price_col']].last().unstack()
        price_panel.rename(columns=cfg['sector_map'], inplace=True)
    else:
        # --- US FORMAT (WIDE) ---
        # Convert the date column and set it as index
        df_raw[cfg['date_col']] = pd.to_datetime(df_raw[cfg['date_col']], errors='coerce')
        df_raw.set_index(cfg['date_col'], inplace=True)
        
        # Rename the wide columns directly based on the sector_map
        price_panel = df_raw.rename(columns=cfg['sector_map'])

    # 3. Final cleaning: ensure numeric and sort
    for col in price_panel.columns:
        price_panel[col] = pd.to_numeric(price_panel[col], errors='coerce')
    price_panel.sort_index(inplace=True)

    # 4. Ensure all GICS sectors exist, even if missing in raw data
    # (Fill missing columns with NaN)
    for sector in GICS_SECTORS:
        if sector not in price_panel.columns:
            price_panel[sector] = np.nan

    # 5. Final Selection
    price_panel = price_panel[GICS_SECTORS]
    log.info('[%s] Price panel ready: %s', MARKET.upper(), price_panel.shape)

2026-06-25 14:01:29  INFO      [CHINA] Price panel ready: (3600, 11)
2026-06-25 14:01:29  INFO      [US] Price panel ready: (3872, 11)


In [20]:
# pandas might fail to find the column names defined in GICS_SECTORS list within the columns of price_panel dataframe
# usually because of mismatch between expected schema and actual schema
# ---------- for debugging ----------
print("Debugging Info:")
print("--------------------")
print("Columns currently in price_panel:", price_panel.columns.tolist())
print("Expected GICS_SECTORS:", GICS_SECTORS)
print("Loading data from:", filepath)
print("Columns in df_raw:", df_raw.columns.tolist())
print("--------------------")
print("End debugging")
# ---------- for debugging ----------

Debugging Info:
--------------------
Columns currently in price_panel: ['Energy', 'Materials', 'Industrials', 'Consumer Discretionary', 'Consumer Staples', 'Health Care', 'Financials', 'Information Technology', 'Communication Services', 'Utilities', 'Real Estate']
Expected GICS_SECTORS: ['Energy', 'Materials', 'Industrials', 'Consumer Discretionary', 'Consumer Staples', 'Health Care', 'Financials', 'Information Technology', 'Communication Services', 'Utilities', 'Real Estate']
Loading data from: /Users/nath/Library/Mobile Documents/com~apple~CloudDocs/Desktop/CSAI_Y02_RESEARCH_INTERNSHIP/Modelling-Industry-Spillover-Effects-Using-Graph-Based-Methods/data/us/US industry index.xlsx
Columns in df_raw: ['S&P500 ES ENERGY - PRICE INDEX', 'S&P500 ES MATERIALS - PRICE INDEX', 'S&P500 ES INDUSTRIALS - PRICE INDEX', 'S&P500 ES CONSUMER DISCRETIONARY - PRICE INDEX', 'S&P500 ES CONSUMER STAPLES - PRICE INDEX', 'S&P500 ES HEALTH CARE - PRICE INDEX', 'S&P500 ES FINANCIALS - PRICE INDEX', 'S&P500 ES

### 2.1 – Load raw prices

Load raw price data for the active MARKET and reshape it into a clean, date-indexed, sector-columned price panel. 

Three input shapes are handled:
- multi-sheet Excel workbooks
- single-sheet long-format Excel
- CSV. 

The resulting `price_panel` has GICS sector names as columns and a sorted `DatetimeIndex`, ready for log-return calculation in the next cell.

In [12]:
cfg      = MARKET_CONFIGS['china']
# cfg      = MARKET_CONFIGS['us']

filepath = DATA_DIR / MARKET / cfg['file']
if not filepath.exists():
    for ext in ['.xlsx', '.xls', '.csv']:
        alt = filepath.with_suffix(ext)
        if alt.exists(): filepath = alt; break

if filepath.suffix in ('.xlsx', '.xls'):
    if cfg['multi_sheet']:
        # Pandas calls openpyxl here, but it can read both .xlsx and .xls
        xl = pd.ExcelFile(filepath)
        panels = {}
        for sheet in xl.sheet_names:
            if sheet not in cfg['sector_map']: continue
            df_s = xl.parse(sheet)
            df_s[cfg['date_col']] = pd.to_datetime(df_s[cfg['date_col']], errors='coerce')
            df_s.set_index(cfg['date_col'], inplace=True)
            df_s = df_s[~df_s.index.isna()]
            panels[cfg['sector_map'][sheet]] = pd.to_numeric(df_s[cfg['price_col']], errors='coerce')
        price_panel = pd.DataFrame(panels)
    else:
        # 1. Read the long-format file
        df_raw = pd.read_excel(filepath)
        
        # 2. Convert Date column to datetime
        df_raw[cfg['date_col']] = pd.to_datetime(df_raw[cfg['date_col']], errors='coerce')
        
        # 3. Needed to swap the rows and columns, 
        # ensures that the columns now consist of the sector names and the rows are indexed by date. 
        # Crucial for time series analysis.
        # Since there are duplicate index codes for the sectors on the rows, 
        # this line is needed to combine them by taking the last available price for each date and sector combination.
        # Helps to solve "duplicate index" error:
        price_panel = df_raw.groupby([cfg['date_col'], 'Index'])[cfg['price_col']].last().unstack()
        
        # 4. Cleanup: Rename numeric indexes to readable sector names
        # Ensure your MARKET_CONFIGS['china']['sector_map'] maps 986 -> 'Energy', etc.
        price_panel.rename(columns=cfg['sector_map'], inplace=True)
        
        # 5. Ensure numeric data and sort
        for col in price_panel.columns:
            price_panel[col] = pd.to_numeric(price_panel[col], errors='coerce')
        price_panel.sort_index(inplace=True)
else:
    df_raw = pd.read_csv(filepath)
    df_raw[cfg['date_col']] = pd.to_datetime(df_raw[cfg['date_col']], errors='coerce')
    df_raw.set_index(cfg['date_col'], inplace=True)
    df_raw = df_raw[~df_raw.index.isna()]
    rename = {k: v for k, v in cfg['sector_map'].items() if k in df_raw.columns}
    df_raw.rename(columns=rename, inplace=True)
    if cfg['benchmark_col'] and cfg['benchmark_col'] in df_raw.columns:
        df_raw.drop(columns=[cfg['benchmark_col']], inplace=True)
    price_panel = df_raw

# pandas might fail to find the column names defined in GICS_SECTORS list within the columns of price_panel dataframe
# usually because of mismatch between expected schema and actual schema
# ---------- for debugging ----------
print("Debugging Info:")
print("Columns currently in price_panel:", price_panel.columns.tolist())
print("Expected GICS_SECTORS:", GICS_SECTORS)
print("Loading data from:", filepath)
print("End debugging")
# ---------- for debugging ----------

price_panel = price_panel[GICS_SECTORS]
price_panel.index = pd.DatetimeIndex(price_panel.index)
price_panel.sort_index(inplace=True)
log.info('[%s] Price panel: %s', MARKET.upper(), price_panel.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/nath/Library/Mobile Documents/com~apple~CloudDocs/Desktop/CSAI_Y02_RESEARCH_INTERNSHIP/Modelling-Industry-Spillover-Effects-Using-Graph-Based-Methods/data/us/China Industry Index.xlsx'

### 2.2 – Percentage log returns `R(i,t) = 100 * ln(P_t/P_{t-1})`

- Zero prices are treated as missing (log is undefined at 0) rather than producing -inf

- Using log-differences instead of simple returns keeps the series time-additive, which is the convention assumed by the GARCH/VAR stages that follow

- Drop only rows where <u>EVERY</u> sector is NaN (e.g. 1st row, which has no prior price to difference against). Rows with partial NaNs are kept so each sector's GARCH fit can later use its own available history.

In [ ]:
log_prices  = np.log(price_panel.replace(0, np.nan))
pct_returns = log_prices.diff() * 100.0
pct_returns.dropna(how='all', inplace=True)


### 2.3 – Quality checks: jump flags & NaN audit

Data-quality audit: 

- flag any `|return|` exceeding `JUMP_THRESHOLD` (likely data error, corporate action, or genuine extreme event)
- log each flagged observation via `_log_issue()` for later review
- Flagged rows are kept in the panel, flagging is informational, not a filter

In [ ]:
jump_mask = pct_returns.abs() > JUMP_THRESHOLD
n_flags   = jump_mask.sum().sum()
if n_flags:
    for col in jump_mask.columns:
        for dt in jump_mask.index[jump_mask[col]]:
            _log_issue(MARKET, 'JUMP', f'{col} {dt.date()} R={pct_returns.loc[dt,col]:.2f}%')
n_nan = pct_returns.isna().sum().sum()
log.info('[%s] Returns: %s obs | %d jump flags | %d NaNs',
         MARKET.upper(), pct_returns.shape, n_flags, n_nan)


2026-06-18 23:08:09  INFO      [CHINA] Returns: (3599, 11) obs | 0 jump flags | 11 NaNs


### 2.4 – Persist

Persist cleaned returns (and any jump flags) to `outputs/<MARKET>/` and print summary statistics as a sanity check before volatility modelling begins

In [ ]:
out_m = OUTPUTS_DIR / MARKET
out_m.mkdir(parents=True, exist_ok=True)
pct_returns.to_csv(out_m / 'pct_returns.csv')
if n_flags: jump_mask.to_csv(out_m / 'jump_flags.csv')

print(pct_returns.describe().round(4))

Index     Energy  Materials  Industrials  Consumer Discretionary  \
count  3598.0000  3598.0000    3598.0000               3598.0000   
mean     -0.0088     0.0095       0.0086                  0.0064   
std       1.6765     1.6555       1.5767                  1.4853   
min      -9.6650    -9.5937     -10.0163                 -9.5276   
25%      -0.8153    -0.7661      -0.7430                 -0.7113   
50%       0.0094     0.0962       0.0381                  0.0406   
75%       0.8473     0.9037       0.8626                  0.8080   
max       7.7097     7.6933       9.4133                  7.8135   

Index  Consumer Staples  Health Care  Financials  Information Technology  \
count         3598.0000    3598.0000   3597.0000               3598.0000   
mean             0.0128       0.0083      0.0158                  0.0357   
std              1.5229       1.5993      1.4822                  1.9692   
min             -9.0625      -8.8301    -10.1755                -10.3574   
25%    

## 3.0 — GARCH / EGARCH Conditional Volatility Panel
Fits GARCH(1,1)-t and EGARCH(1,1)-t to each of 11 GICS sectors. 
Convergence failures are retried with extended iterations; persistent failures produce NaN columns.

`GARCH(1,1)-t / EGARCH(1,1)-t` fitting stage. 

For each of the 11 GICS sectors:
- fit a constant-mean GARCH and EGARCH model with Student-t innovations to the percentage log returns
- extract the conditional volatility series 
- sanity-check the estimated parameters

`EXPECTED_BOUNDS` gives the typical empirical range for daily-frequency equity volatility parameters, i.e. values outside this range are flagged (not rejected), since genuinely turbulent markets can produce legitimate outlier estimates.

### 3.1 _check_params(params, model_type, sector)
Validate fitted GARCH/EGARCH parameters against expected economic bounds.

Computes volatility persistence (alpha + beta) and flags estimates that are explosive (persistence >= 1, implying a non-stationary variance process), unusually low/high relative to PERSISTENCE_LOW/PERSISTENCE_HIGH, or have a non-positive omega (which makes the unconditional variance ill-defined).

Args:
- params (dict): Fitted parameters as returned by `res.params.to_dict()` from an arch_model fit (e.g. "omega", "alpha[1]", "beta[1]")
- model_type (str): Model family the parameters came from ("Garch" or "EGARCH"). Not currently used inside the function body, but kept for call-site clarity.
- sector (str): GICS sector name the parameters belong to.

Returns:
- tuple[list[str], float]: A list of human-readable warning strings (empty if every check passes) and the computed persistence (alpha + beta), or NaN if alpha/beta could not be extracted. Persistence = alpha + beta: the rate at which a volatility shock decays. Values close to 1 imply long memory (shocks fade slowly); values >= 1 imply a non-stationary (explosive) variance process.
### 3.2 _fit_model(series, sector, vol_type='Garch')
Generic fitter for GARCH or EGARCH; returns (vol_series, param_dict, converged).

Args:
- series (pandas.Series): Percentage log returns for a single sector.
- sector (str): GICS sector name, used for logging/labelling output.
- vol_type (str): Volatility model to fit -- "Garch" or "EGARCH".
        Defaults to "Garch".

Returns:
- vol_series (pandas.Series | None): Conditional volatility series indexed like `series`, or None if fitting failed or there was insufficient data.
- param_dict (dict): Estimated parameters, persistence, fit diagnostics (log-likelihood, AIC) and any validation warnings. Always returned, even on failure, with a 'note' explaining why
- converged (bool): Whether the optimizer reported convergence

Raises:
- No exceptions propagate out of this function, optimizer failures are caught internally and reported via the returned values instead
### 3.3 build_panel(pct_returns, vol_type)
Fit a volatility model across every GICS sector and assemble the panel.

Loops over GICS_SECTORS, fits `vol_type` ('Garch' or 'EGARCH') to each sector's return series via `_fit_model`, and collects the resulting conditional-volatility series and parameter diagnostics into two aligned tables. Sectors missing from `pct_returns` are filled with NaN volatility rather than dropped, so the output panel always has all 11 columns.

Args:
- pct_returns (pandas.DataFrame): Percentage log returns, one column per GICS sector
- vol_type (str): Volatility model to fit for every sector, "Garch" or "EGARCH"

Returns (tuple[pandas.DataFrame, pandas.DataFrame]):
- vol_panel: Conditional volatility, one column per sector, sorted by date 
- param_table: One row per sector with fitted parameters, persistence, fit diagnostics, and convergence status, indexed by sector name

In [21]:
EXPECTED_BOUNDS = {'omega': (1e-8, 10.), 'alpha': (0.01, 0.25),
                   'beta': (0.70, 0.99), 'gamma': (-1., 0.)}

def _check_params(params, model_type, sector):
    warns = []
    alpha = params.get('alpha[1]', params.get('alpha', np.nan))
    beta  = params.get('beta[1]',  params.get('beta',  np.nan))
    omega = params.get('omega', np.nan)
    persist = alpha + beta if not (np.isnan(alpha) or np.isnan(beta)) else np.nan
    if not np.isnan(persist) and persist >= 1.: warns.append(f'alpha+beta={persist:.4f}>=1')
    if not np.isnan(persist) and persist < PERSISTENCE_LOW:
        warns.append(f'low persistence={persist:.4f}')
    if not np.isnan(persist) and persist > PERSISTENCE_HIGH:
        warns.append(f'high persistence={persist:.4f}')
    if not np.isnan(omega) and omega <= 0: warns.append(f'omega={omega:.2e}<=0')
    return warns, persist

def _fit_model(series, sector, vol_type='Garch'):
    """Generic fitter for GARCH or EGARCH; returns (vol_series, param_dict, converged)."""
    # Force clean dropping of missing values safely
    s = series.dropna()
    if len(s) < 100:
        return None, {'sector': sector, 'converged': False, 'note': 'insufficient data'}, False
        
    for attempt, max_it in enumerate([MAX_ITER, MAX_ITER_RETRY], 1):
        try:
            am  = arch_model(s, vol=vol_type, p=1, q=1, dist=INNOVATION_DIST, mean='Constant')
            res = am.fit(disp='off', show_warning=False, options={'maxiter': max_it})
            conv = res.convergence_flag == 0
            if not conv and attempt == 1: continue
            
            params = res.params.to_dict()
            warns, persist = _check_params(params, vol_type, sector)
            alpha = params.get('alpha[1]', np.nan)
            beta  = params.get('beta[1]',  np.nan)
            rec = {'sector': sector,
                   'omega':       round(params.get('omega', np.nan), 8),
                   'alpha':       round(alpha, 6),
                   'beta':        round(beta,  6),
                   'persistence': round(persist, 6) if not np.isnan(persist) else np.nan,
                   'nu':          round(params.get('nu', np.nan), 4),
                   'log_lik':     round(res.loglikelihood, 4),
                   'aic':         round(res.aic, 4),
                   'converged':   conv,
                   'warnings':    '; '.join(warns) if warns else 'OK'}
            if vol_type == 'EGARCH':
                gamma = params.get('gamma[1]', np.nan)
                rec['gamma']    = round(gamma, 6)
                rec['leverage'] = 'YES' if (not np.isnan(gamma) and gamma < 0) else 'NO'
            vol_s = pd.Series(res.conditional_volatility, index=s.index, name=sector)
            return vol_s, rec, conv
        except Exception as exc:
            if attempt == 2:
                return None, {'sector': sector, 'converged': False, 'note': str(exc)}, False
    return None, {'sector': sector, 'converged': False, 'note': 'all attempts failed'}, False

def build_panel(pct_returns, vol_type, current_market):
    vols, params = {}, []
    
    # Dynamically find which sectors actually exist in this specific DataFrame
    # This prevents matching errors if renaming left them as raw column strings
    available_sectors = [s for s in GICS_SECTORS if s in pct_returns.columns]
    
    # Fallback to whatever string headers exist if mapping completely missed
    if not available_sectors:
        available_sectors = list(pct_returns.columns)
        
    for sec in available_sectors:
        # Avoid running on entirely null/blank index series
        if pct_returns[sec].isna().all():
            vols[sec] = pd.Series(np.nan, index=pct_returns.index)
            params.append({'sector': sec, 'converged': False, 'note': 'column is all NaN'})
            continue
            
        vs, rec, _ = _fit_model(pct_returns[sec], sec, vol_type)
        vols[sec]   = vs if vs is not None else pd.Series(np.nan, index=pct_returns.index)
        params.append(rec)
        log.info('[%s] %s %s  persist=%.4f  conv=%s',
                 current_market.upper(), vol_type, sec, rec.get('persistence', np.nan), rec.get('converged'))
                 
    vol_panel   = pd.DataFrame(vols).sort_index()
    param_table = pd.DataFrame(params).set_index('sector')
    return vol_panel, param_table

# Initialize master storage matrices
garch_panels = {}
egarch_panels = {}

for MARKET in MARKETS:
    print(f"\n=========================================\nRUNNING VOLATILITY MODELING FOR: {MARKET.upper()}\n=========================================")
    
    pct_returns = all_returns[MARKET]
    out_m = OUTPUTS_DIR / MARKET
    out_m.mkdir(parents=True, exist_ok=True)
    
    # Pass MARKET variable explicitly down to prevent naming overlaps
    log.info('[%s] Fitting GARCH(1,1)-t ...', MARKET.upper())
    garch_panel,  garch_params  = build_panel(pct_returns, 'Garch', MARKET)

    log.info('[%s] Fitting EGARCH(1,1)-t ...', MARKET.upper())
    egarch_panel, egarch_params = build_panel(pct_returns, 'EGARCH', MARKET)

    # Persist individual market results to their respective folders
    garch_panel.to_csv(out_m  / 'garch_volatility_panel.csv')
    egarch_panel.to_csv(out_m / 'egarch_volatility_panel.csv')
    garch_params.to_csv(out_m  / 'garch_parameters.csv')
    egarch_params.to_csv(out_m / 'egarch_parameters.csv')

    print(f'\n── [{MARKET.upper()}] GARCH Parameters ──')
    display(garch_params)
    print(f'\n── [{MARKET.upper()}] EGARCH Parameters ──')
    display(egarch_params)
    
    garch_panels[MARKET] = garch_panel
    egarch_panels[MARKET] = egarch_panel


RUNNING VOLATILITY MODELING FOR: CHINA


NameError: name 'all_returns' is not defined

In [ ]:
def extract_top_connections(gfevd_matrix, sectors, top_n=TOP_N_CONNS):
    """
    Clears the main diagonal of a GFEVD matrix and captures the 
    top N directional spillover edges across sectors for network graphs.
    """
    import pandas as pd
    df_mat = pd.DataFrame(gfevd_matrix, index=sectors, columns=sectors)
    
    connections = []
    for source in sectors:
        for target in sectors:
            if source == target:
                continue
            weight = df_mat.loc[source, target]
            connections.append({
                'Source': source,
                'Target': target,
                'Weight': weight
            })
            
    return pd.DataFrame(connections).sort_values(by='Weight', ascending=False).head(top_n)

log.info("Network extractor utility function initialized.")

2026-06-24 23:58:46  INFO      Network extractor utility function initialized.


## 4.0 — ADF Stationarity & VAR Analysis
ADF test on all 11 GARCH volatility series, then VAR(p) with AIC lag selection (p ∈ 1…MAX_LAGS).

ADF stationarity testing + VAR(p) fitting on the GARCH conditional volatility panel. The Diebold-Yilmaz framework requires a stationary VAR, so each sector's volatility series is checked for a unit root before the VAR is fit. Lag order p is then chosen by AIC over MAX_LAGS candidates.

ADF test, one sector at a time: H0 is "series has a unit root" (non-stationary). Rejecting H0 (p < ADF_PVAL_THR) supports using the series directly in a level VAR without differencing.

In [ ]:
# ── ADF stationarity ──────────────────────────────────────────────────────────
vol_panel = garch_panel.copy()   # in-memory from previous cell

adf_records = []
for sec in vol_panel.columns:
    s = vol_panel[sec].dropna()
    try:
        stat, pval, *_ = adfuller(s, maxlag=ADF_MAXLAG, autolag='AIC')
        stationary = bool(pval < ADF_PVAL_THR)
    except Exception as e:
        stat, pval, stationary = np.nan, np.nan, False
        _log_issue(MARKET, 'ADF', f'{sec}: {e}')
    if not stationary:
        _log_issue(MARKET, 'ADF', f'{sec} p={pval:.4f} — unit root not rejected')
    adf_records.append({'sector': sec,
                        'adf_stat':   round(stat,  4) if not np.isnan(stat)  else np.nan,
                        'p_value':    round(pval,  4) if not np.isnan(pval)  else np.nan,
                        'stationary': stationary})

adf_table = pd.DataFrame(adf_records).set_index('sector')
print('── ADF Stationarity ──')
display(adf_table)

# –––––––––– fixes for failed lag selection ––––––––––

# The NaN/Inf issue: 
# VAR().select_order and .fit() perform matrix inversions. 
#   If a single NaN exists, the resulting matrix is undefined. 
#   If an inf exists, the math breaks.

# The DLASCL error: 
# This is a low-level error from LAPACK (the library statsmodels uses for matrix algebra). 
# It is triggered when you attempt to perform operations on a matrix that contains NaNs or Infs, 
# causing the internal scaling algorithms to fail

# For China industry Index.xlsx, only 3 rows were dropped
print(f"Panel size before cleaning: {vol_panel.shape}")

# Clean vol_panel for VAR 
# Drop rows that have ANY NaN values across the sectors
vol_panel = vol_panel.dropna()

# if all 0s, the model will run successfully
print("Missing values per column:\n", vol_panel.isna().sum())

# Double check for infinite values (common in volatility modeling)
vol_panel = vol_panel.replace([np.inf, -np.inf], np.nan).dropna()

print(f"Panel size after cleaning: {vol_panel.shape}")
# –––––––––– fixes for failed lag selection ––––––––––

# ── AIC lag selection ─────────────────────────────────────────────────────────
# AIC lag selection minimises information loss vs. model complexity over
# candidate lag orders p = 1..MAX_LAGS. If selection itself fails (e.g. due
# to a near-singular covariance matrix), fall back to a conservative p=2
# rather than letting the whole pipeline crash.
try:
    lag_results = VAR(vol_panel).select_order(maxlags=MAX_LAGS)
    lag_order   = int(lag_results.aic)
    log.info('[%s] AIC lag order: p=%d', MARKET.upper(), lag_order)
    print(lag_results.summary())
except Exception as e:
    lag_order = 2
    _log_issue(MARKET, 'LAG_SELECTION', f'AIC failed — fallback p={lag_order}: {e}')
    log.warning('[%s] Lag selection failed; using p=%d', MARKET.upper(), lag_order)

# ── Fit VAR(p) ────────────────────────────────────────────────────────────────
# Fit the VAR(p) on volatility LEVELS (not returns): this is the model the
# Diebold-Yilmaz GFEVD / connectedness measures in the next cell are derived
# from -- it captures how a volatility shock in one sector propagates to others.
var_result = VAR(vol_panel).fit(maxlags=lag_order, ic=None)
log.info('[%s] VAR(%d) fitted. Log-likelihood: %.2f',
         MARKET.upper(), lag_order, var_result.llf)

# ── Persist ───────────────────────────────────────────────────────────────────
with open(out_m / 'var_result.pkl', 'wb') as fh: pickle.dump(var_result, fh)
adf_table.to_csv(out_m / 'adf_stationarity.csv')
with open(out_m / 'var_lag_order.txt', 'w') as fh:
    fh.write(f'Market: {MARKET.upper()}\nLag order (AIC): {lag_order}\nTimestamp: {datetime.now().isoformat()}\n')
print(f'\nVAR({lag_order}) fit complete.')

── ADF Stationarity ──


,adf_stat,p_value,stationary
sector,,,
Energy,-6.0416,0.0,True
Materials,-6.9892,0.0,True
Industrials,-6.3198,0.0,True
Consumer Discretionary,-5.6504,0.0,True
Consumer Staples,-6.6926,0.0,True
Health Care,-5.8765,0.0,True
Financials,-5.3340,0.0,True
Information Technology,-5.3912,0.0,True
Communication Services,-5.9869,0.0,True


2026-06-18 23:22:03  INFO      [CHINA] AIC lag order: p=2
2026-06-18 23:22:03  INFO      [CHINA] VAR(2) fitted. Log-likelihood: 55964.03


Panel size before cleaning: (3599, 11)
Missing values per column:
 Energy                    0
Materials                 0
Industrials               0
Consumer Discretionary    0
Consumer Staples          0
Health Care               0
Financials                0
Information Technology    0
Communication Services    0
Utilities                 0
Real Estate               0
dtype: int64
Panel size after cleaning: (3596, 11)
 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -31.98      -31.96   1.296e-14      -31.97
1       -62.21     -61.98*   9.595e-28     -62.13*
2      -62.21*      -61.78  9.567e-28*      -62.06
3       -62.21      -61.57   9.604e-28      -61.98
4       -62.20      -61.35   9.658e-28      -61.90
5       -62.19      -61.13   9.797e-28      -61.81
6       -62.17      -60.90   1.001e-27      -61.72
7       -62.14      -60.66   1.027e-27      -61.62
8       -62.1

## 5.0 — Diebold-Yilmaz Static Network & Rolling Connectedness Index
KPPS Generalised FEVD → row-normalised adjacency matrix → TO/FROM/NET measures → TSI.
Rolling TSI re-estimates VAR inside each window of `ROLLING_WINDOW` days.

### 5.1 compute_gfevd(var_res, H)
Generalised Forecast Error Variance Decomposition (KPPS, order-invariant).

Implements the Diebold-Yilmaz (2014) connectedness measure: for each sector i, decomposes the H-step-ahead forecast error variance into the share attributable to shocks originating in each sector j. Unlike the Cholesky-based FEVD, the KPPS GFEVD does not depend on variable ordering.

Args:
- var_res (statsmodels.tsa.vector_ar.var_model.VARResultsWrapper): A fitted VAR(p) result (e.g. from `VAR(vol_panel).fit(...)`).
- H (int): Forecast horizon. Impulse responses are evaluated for h = 0, 1, ..., H (H + 1 steps total, including the contemporaneous shock).

Returns:
- numpy.ndarray: An (n, n) raw (pre-normalisation) GFEVD matrix theta, where theta[i, j] is the share of sector i's H-step forecast-error variance explained by shocks to sector j. Rows do not yet sum to 1 (see `row_norm` for the normalised version theta*).

### 5.2 row_norm(theta)
Row-normalise a GFEVD matrix so each row sums to 1 (theta*).

The raw KPPS GFEVD matrix does not sum to 1 across each row because the shocks are not orthogonalised. Dividing each row by its own sum produces theta*, the normalised matrix used for all directional spillover measures (TO, FROM, NET, the Total Spillover Index).

Args:
- theta (numpy.ndarray): Raw (n, n) GFEVD matrix from `compute_gfevd`

Returns:
- numpy.ndarray: Row-normalised (n, n) matrix theta*. Rows that sum to zero are left unchanged (divided by 1) to avoid a 0/0 division

### 5.3 directional_measures(th_star, names)
Compute Diebold-Yilmaz directional spillover measures from theta*.

Args:
- th_star (numpy.ndarray): Row-normalised (n, n) GFEVD matrix, as returned by `row_norm`.
- names (list[str]): Sector names, in the same order as the rows/columns of `th_star`

Returns (tuple[pandas.DataFrame, float]):
- df: One row per sector with TO (spillovers sent to all other sectors), FROM (spillovers received), NET (TO - FROM), and OWN (own-shock variance share), all expressed in percentage
- tsi (float): The Total Spillover Index is the share of total forecast-error variance attributable to cross-sector spillovers, in percentage

In [ ]:
sector_names = list(vol_panel.columns)

# Debug check
irf_test = var_result.irf(periods=HORIZON)
print("Type of irf_test.irfs:", type(irf_test.irfs))
# If this is a list of (n,n) arrays, the conversion to np.array() is correct.

"""
# ── GFEVD (KPPS, order-invariant) ────────────────────────────────────────────
# The Indexing Error: 
    # Previous code tried to access A[h][i] (which implies A is a 3D array or a list of arrays). 
    # statsmodels IRF objects often return their internal storage as a list of matrices. 
    # The InvalidIndexError happened because the object the model was slicing 
    # didn't support the multi-dimensional slicing [h][i, 0] the model was performing.
    # 
    # np.asarray(): 
    # By wrapping the irfs in np.asarray(), the data is forced into a standard NumPy structure.
    # Even if statsmodels returns a subclass or a Pandas-indexed array, 
    # this strips away the index labels that were likely causing Pandas to try and look up a key (like 0) 
    # instead of using it as a position. 
    # 
    # A[h, i, :]: 
    # This is the "proper" 3D indexing syntax for NumPy. 
    # Using [h][i] in the previous code creates two separate index operations. 
    # If A[h] were a Pandas object, [i] would trigger a label lookup. 
    # A[h, i, :] tells NumPy to perform the access in one single operation, which is faster and safer.
# 
# The Horizon: 
# Note that periods=H in statsmodels typically includes the contemporaneous shock (lag 0) up to lag H. 
# The loop range(H) might have been missing the $h=0$ impact. The range is updated to H + 1.
# 
# Note on range(H+1): 
# Ensure HORIZON variable is set correctly. 
# In Diebold-Yilmaz frameworks, the horizon $H$ is usually 10 days. 
# If HORIZON is 10, range(H+1) covers 0 to 10 (11 steps total).
"""
def compute_gfevd(var_res, H):
    # Ensure irfs is a numpy array (H+1, n, n)
    # Using np.asarray ensures we are dealing with pure numpy indexing
    A = np.asarray(var_res.irf(periods=H).irfs)
    Sig = np.asarray(var_res.sigma_u)
    n = Sig.shape[0]
    sig_diag = np.diag(Sig)
    theta = np.zeros((n, n))
    
    for i in range(n):
        # Calculate denominator: sum of (row_i @ Sig @ row_i.T) across H
        # A[h, i, :] is the i-th row at horizon h
        denom = sum(A[h, i, :] @ Sig @ A[h, i, :].T for h in range(H + 1))
        
        for j in range(n):
            # Calculate numerator: (1/sigma_jj) * sum((row_i @ Sig_column_j)^2)
            numer = (1. / sig_diag[j]) * sum((A[h, i, :] @ Sig[:, j])**2 for h in range(H + 1))
            
            theta[i, j] = numer / denom if denom > 1e-10 else 0.
            
    return theta

def row_norm(theta):
    rs = theta.sum(axis=1, keepdims=True)
    return theta / np.where(rs == 0, 1., rs)

def directional_measures(th_star, names):
    off = th_star.copy(); np.fill_diagonal(off, 0.)
    TO   = off.sum(axis=0)
    FROM = off.sum(axis=1)
    NET  = TO - FROM
    OWN  = np.diag(th_star)
    tsi  = off.sum() / len(names) * 100
    df   = pd.DataFrame({'TO': TO*100, 'FROM': FROM*100,
                         'NET': NET*100, 'OWN': OWN*100},
                        index=names).round(4)
    return df, tsi

# ── Full-sample static connectedness ─────────────────────────────────────────
# error here:
theta_raw  = compute_gfevd(var_result, HORIZON)
theta_star = row_norm(theta_raw)
measures, tsi = directional_measures(theta_star, sector_names)

print(f'Total Spillover Index (H={HORIZON}, full sample): {tsi:.2f}%')
display(measures)

# ── Persist static outputs ────────────────────────────────────────────────────
cm_df = pd.DataFrame(theta_star, index=sector_names, columns=sector_names)
cm_df.to_csv(out_m / 'connectedness_matrix.csv')
measures.to_csv(out_m / 'directional_measures.csv')
with open(out_m / 'tsi_fullsample.txt', 'w') as fh:
    fh.write(f'Market: {MARKET.upper()}\nTSI (H={HORIZON}): {tsi:.4f}%\nTimestamp: {datetime.now().isoformat()}\n')

# ── Rolling connectedness ─────────────────────────────────────────────────────
# Walk-forward estimation: for each window end-date, re-fit the VAR on the
# trailing ROLLING_WINDOW observations only (using the SAME lag order chosen
# on the full sample), recompute GFEVD/TSI for that window, and slide the
# window forward by ROLLING_STEP. This traces how connectedness evolves over
# time rather than reporting a single full-sample snapshot.
dates    = vol_panel.index
n_obs    = len(dates)
records  = []
n_ok = n_fail = 0

log.info('[%s] Rolling: w=%d, step=%d, lag=%d, H=%d',
         MARKET.upper(), ROLLING_WINDOW, ROLLING_STEP, lag_order, HORIZON)

for end_idx in range(ROLLING_WINDOW - 1, n_obs, ROLLING_STEP):
    s_date = dates[end_idx - ROLLING_WINDOW + 1]
    e_date = dates[end_idx]
    win    = vol_panel.loc[s_date:e_date]
    if len(win) < ROLLING_WINDOW: continue
    try:
        vr   = VAR(win).fit(maxlags=lag_order, ic=None)
        th   = row_norm(compute_gfevd(vr, HORIZON))
        meas, tsi_w = directional_measures(th, sector_names)
        row  = {'date': e_date, 'TSI': round(tsi_w, 4)}
        for sec in sector_names:
            row[f'NET_{sec}']  = meas.loc[sec, 'NET']
            row[f'TO_{sec}']   = meas.loc[sec, 'TO']
            row[f'FROM_{sec}'] = meas.loc[sec, 'FROM']
        records.append(row)
        n_ok += 1
    except Exception as exc:
        n_fail += 1
        _log_issue(MARKET, 'ROLLING', f'{s_date.date()}→{e_date.date()}: {exc}')
    if n_ok % 100 == 0 and n_ok:
        log.info('[%s] Rolling: %d windows done (end=%s)',
                 MARKET.upper(), n_ok, e_date.date())

log.info('[%s] Rolling complete: %d ok, %d failed', MARKET.upper(), n_ok, n_fail)

if records:
    rolling_df = pd.DataFrame(records).set_index('date')
    rolling_df.index = pd.DatetimeIndex(rolling_df.index)
    rolling_df.to_csv(out_m / 'rolling_tsi.csv')
    print(f'\nRolling TSI: {len(rolling_df)} windows saved.')
    display(rolling_df[['TSI']].tail(10))
else:
    print('No rolling windows produced output.')

Type of irf_test.irfs: <class 'numpy.ndarray'>
Total Spillover Index (H=10, full sample): 81.26%


,TO,FROM,NET,OWN
Energy,65.0889,79.4884,-14.3995,20.5116
Materials,97.3252,84.3496,12.9757,15.6504
Industrials,103.4363,86.1185,17.3178,13.8815
Consumer Discretionary,103.9348,85.3391,18.5957,14.6609
Consumer Staples,79.4265,78.1582,1.2683,21.8418
Health Care,72.1515,81.6202,-9.4687,18.3798
Financials,58.0908,74.0343,-15.9434,25.9657
Information Technology,78.9361,83.3813,-4.4452,16.6187
Communication Services,77.9074,82.1064,-4.1989,17.8936
Utilities,84.5947,80.1689,4.4258,19.8311


2026-06-18 23:43:31  INFO      [CHINA] Rolling: w=200, step=1, lag=2, H=10
2026-06-18 23:43:31  INFO      [CHINA] Rolling: 100 windows done (end=2012-10-26)
2026-06-18 23:43:32  INFO      [CHINA] Rolling: 200 windows done (end=2013-03-27)
2026-06-18 23:43:32  INFO      [CHINA] Rolling: 300 windows done (end=2013-08-26)
2026-06-18 23:43:33  INFO      [CHINA] Rolling: 400 windows done (end=2014-01-23)
2026-06-18 23:43:34  INFO      [CHINA] Rolling: 500 windows done (end=2014-06-25)
2026-06-18 23:43:34  INFO      [CHINA] Rolling: 600 windows done (end=2014-11-20)
2026-06-18 23:43:35  INFO      [CHINA] Rolling: 700 windows done (end=2015-04-21)
2026-06-18 23:43:35  INFO      [CHINA] Rolling: 800 windows done (end=2015-09-14)
2026-06-18 23:43:36  INFO      [CHINA] Rolling: 900 windows done (end=2016-02-16)
2026-06-18 23:43:36  INFO      [CHINA] Rolling: 1000 windows done (end=2016-07-11)
2026-06-18 23:43:37  INFO      [CHINA] Rolling: 1100 windows done (end=2016-12-07)
2026-06-18 23:43:37  


Rolling TSI: 3397 windows saved.


,TSI
date,
2026-05-19,57.6892
2026-05-20,57.0444
2026-05-21,56.4995
2026-05-22,55.1256
2026-05-25,55.1377
2026-05-26,54.8689
2026-05-27,54.8157
2026-05-28,54.5463
2026-05-29,54.6212
